# ReadyAI CHiRPE ONNX Summarization + Segmented Inference

This notebook is a variant of `04_readyai_segmented_inference_contract.ipynb`.

The difference is the responsibility boundary. In notebook 04, only per-segment
classification runs inside ReadyAI; segmentation, summarization, and aggregation
run outside. Here, the **entire pipeline runs inside ReadyAI**: ReadyAI ingests
the raw transcript JSON, segments it, summarizes each segment with **Phi-3 ONNX**,
classifies each segment, and aggregates the results into a transcript-level
decision.

Raw transcript JSON goes in, and the aggregated CHR-P decision comes out — all
within ReadyAI. The cells below are labelled by pipeline stage; every stage is
ReadyAI-owned.

## Resource Limits (Optional)

The Phi-3 ONNX summarizer runs **in-process** here (via `onnxruntime-genai`),
not in a Triton container, so Docker flags like `--cpus`/`--memory` do not apply
directly. The cell below exposes the in-process equivalents as simple parameters:

- `PHI3_NUM_THREADS` — caps ONNX Runtime CPU threads (the real analogue of
  `--cpus 3.0`/`4.0`). Set to `None` to use all cores.
- `PHI3_MAX_MEMORY_GB` — optional hard memory ceiling for the **whole kernel**
  (Linux only, via `RLIMIT_AS`). Set to `None` to leave memory uncapped.

Run this cell **first**, before importing/loading any model — the thread setting
is read when ONNX Runtime initializes.

In [1]:
# --- Resource limits (set these, then run the notebook top to bottom) ---
PHI3_NUM_THREADS = 4          # CPU cap, like `--cpus 4.0`. Use None for all cores.
PHI3_MAX_MEMORY_GB = None     # In-process address-space cap (advanced; see note below).

import os

if PHI3_NUM_THREADS is not None:
    # ONNX Runtime / OpenMP read these at session init, so set them before loading.
    os.environ["OMP_NUM_THREADS"] = str(PHI3_NUM_THREADS)
    os.environ["ORT_INTRA_OP_NUM_THREADS"] = str(PHI3_NUM_THREADS)
    print(f"CPU threads capped at {PHI3_NUM_THREADS}")
else:
    print("CPU threads: unlimited (all cores)")

# NOTE on memory: a true RSS ceiling (the real `--memory 8Gi`) must be applied
# when the kernel is launched, because cgroups limit *resident* memory. Run e.g.:
#   systemd-run --user --scope -p CPUQuota=400% -p MemoryMax=8G \
#       conda run --no-capture-output -n chirp jupyter lab
# The in-process knob below uses RLIMIT_AS, which caps *virtual* address space.
# torch + SHAP + ONNX Runtime over-reserve virtual memory far beyond what they
# physically use, so a low value here false-trips on load. Leave it None unless
# you understand that trade-off; prefer the launch-time cgroup cap above.
if PHI3_MAX_MEMORY_GB is not None:
    try:
        import resource

        max_bytes = int(PHI3_MAX_MEMORY_GB * 1024**3)
        _, hard = resource.getrlimit(resource.RLIMIT_AS)
        new_hard = max_bytes if hard == resource.RLIM_INFINITY else min(max_bytes, hard)
        resource.setrlimit(resource.RLIMIT_AS, (max_bytes, new_hard))
        print(f"Virtual-memory (RLIMIT_AS) capped at {PHI3_MAX_MEMORY_GB} GiB")
    except (ImportError, ValueError, OSError) as exc:
        print(f"Could not set memory cap ({exc}); leaving memory uncapped.")
else:
    print("Memory: uncapped in-process (use a launch-time cgroup cap for a true limit)")

CPU threads capped at 4
Memory: uncapped in-process (use a launch-time cgroup cap for a true limit)


## Responsibility Boundary

The entire pipeline runs **inside ReadyAI**: ReadyAI receives the raw transcript
JSON and returns the aggregated transcript-level result. No step is delegated
outside.

| Step | Location |
| --- | --- |
| Raw transcript ingestion | Inside ReadyAI |
| Raw transcript segmentation | Inside ReadyAI |
| **Segment summarization (Phi-3 ONNX)** | **Inside ReadyAI** |
| CHiRPE model inference on each segment | Inside ReadyAI |
| Transcript-level voting or aggregation | Inside ReadyAI |

Raw transcript JSON goes in, and the aggregated CHR-P decision comes out — all
within ReadyAI.

## Inside ReadyAI: Raw Transcript Input

The payload below is the raw transcript that ReadyAI ingests. Unlike notebook 04,
ReadyAI receives this raw transcript directly and owns every step from here on.

In [2]:
raw_transcript_payload = {
    "participant_id": "demo_001",
    "transcript": [
        {"speaker": "interviewer", "text": "Tell me about whether events have had special meaning for you."},
        {"speaker": "interviewee", "text": "Sometimes I feel like television messages are personally directed at me."},
        {"speaker": "interviewer", "text": "Tell me whether people are talking about you or watching you."},
        {"speaker": "interviewee", "text": "At times I worry strangers are watching me when I am outside, but it comes and goes."},
        {"speaker": "interviewer", "text": "Tell me about any trouble concentrating or focusing."},
        {"speaker": "interviewee", "text": "I have mild trouble focusing at school, but I do not get lost or confused about where I am."},
    ],
}

raw_transcript_payload

{'participant_id': 'demo_001',
 'transcript': [{'speaker': 'interviewer',
   'text': 'Tell me about whether events have had special meaning for you.'},
  {'speaker': 'interviewee',
   'text': 'Sometimes I feel like television messages are personally directed at me.'},
  {'speaker': 'interviewer',
   'text': 'Tell me whether people are talking about you or watching you.'},
  {'speaker': 'interviewee',
   'text': 'At times I worry strangers are watching me when I am outside, but it comes and goes.'},
  {'speaker': 'interviewer',
   'text': 'Tell me about any trouble concentrating or focusing.'},
  {'speaker': 'interviewee',
   'text': 'I have mild trouble focusing at school, but I do not get lost or confused about where I am.'}]}

## Inside ReadyAI: Segment the Raw Transcript

This cell performs **segmentation inside ReadyAI**. It uses `SymptomSegmenter`
directly so the payload carries each segment's raw `text`, which ReadyAI will
summarize in the next step.

Unmapped segments are dropped, mirroring `TranscriptPreprocessor`.

In [3]:
from chirpe.data.segmentation import SymptomSegmenter

segmenter = SymptomSegmenter(threshold=0.80)
segments = segmenter.segment_transcript(raw_transcript_payload["transcript"])

mapped_segments = [seg for seg in segments if seg.domain != "unmapped"]
if not mapped_segments:
    raise ValueError("Segmentation produced no mapped segments; check the prompts and threshold.")

participant_id = raw_transcript_payload["participant_id"]
segmented_payload = {
    "participant_id": participant_id,
    "segments": [
        {
            "segment_id": f"{participant_id}_seg_{index + 1:03d}",
            "domain": segment.domain,
            "text": segment.get_text(),
            "start_idx": segment.start_idx,
            "end_idx": segment.end_idx,
        }
        for index, segment in enumerate(mapped_segments)
    ],
}

segmented_payload

{'participant_id': 'demo_001',
 'segments': [{'segment_id': 'demo_001_seg_001',
   'domain': 'P1_Unusual_Thoughts',
   'text': 'Tell me about whether events have had special meaning for you. Sometimes I feel like television messages are personally directed at me.',
   'start_idx': 0,
   'end_idx': 1},
  {'segment_id': 'demo_001_seg_002',
   'domain': 'P2_Suspiciousness',
   'text': 'Tell me whether people are talking about you or watching you. At times I worry strangers are watching me when I am outside, but it comes and goes.',
   'start_idx': 2,
   'end_idx': 3},
  {'segment_id': 'demo_001_seg_003',
   'domain': 'P4_Disoriented',
   'text': 'Tell me about any trouble concentrating or focusing. I have mild trouble focusing at school, but I do not get lost or confused about where I am.',
   'start_idx': 4,
   'end_idx': 5}]}

## Internal Segmented Representation

The object below is the intermediate payload ReadyAI builds after segmentation:
a participant ID plus an ordered list of segments, each carrying **raw segment
text** (not a summary). It is passed to the summarization + classification steps
that follow, all inside ReadyAI.

In [4]:
import json

print(json.dumps(segmented_payload, indent=2))

{
  "participant_id": "demo_001",
  "segments": [
    {
      "segment_id": "demo_001_seg_001",
      "domain": "P1_Unusual_Thoughts",
      "text": "Tell me about whether events have had special meaning for you. Sometimes I feel like television messages are personally directed at me.",
      "start_idx": 0,
      "end_idx": 1
    },
    {
      "segment_id": "demo_001_seg_002",
      "domain": "P2_Suspiciousness",
      "text": "Tell me whether people are talking about you or watching you. At times I worry strangers are watching me when I am outside, but it comes and goes.",
      "start_idx": 2,
      "end_idx": 3
    },
    {
      "segment_id": "demo_001_seg_003",
      "domain": "P4_Disoriented",
      "text": "Tell me about any trouble concentrating or focusing. I have mild trouble focusing at school, but I do not get lost or confused about where I am.",
      "start_idx": 4,
      "end_idx": 5
    }
  ]
}


## Inside ReadyAI: Load the Phi-3 ONNX Summarizer

This is the start of the summarization stage. ReadyAI loads the local Phi-3 ONNX
Runtime GenAI summarizer. The model is loaded once and reused across segments.

Requires the `onnx-llm` extra (`pip install -e ".[onnx-llm]"`). If the model is
not present locally, set `phi3_download=True` or pre-download it with
`scripts/experiments/phi3_onnx_test.py --download`.

The `TOKENIZER_BACKEND` toggle below selects the tokenizer: `"og"`
(onnxruntime-genai, default) or `"hf"` (transformers `AutoTokenizer`).
`og.Generator` runs the model either way; only the tokenizer changes, and the two
produce identical token IDs on clinical text
(see `scripts/experiments/tokenizer_parity_check.py`).

**Generation settings.** Decoding is **greedy and deterministic** — `do_sample=False`
(pinned in `Phi3OnnxSummarizer.summarize_segment`) with the model's default
`num_beams=1`, so generation always takes the argmax token. The remaining
sampling parameters are therefore inert and no repetition controls are applied:

| Parameter | Value | Effect |
| --- | --- | --- |
| `do_sample` | `False` | Greedy decoding (deterministic) |
| `num_beams` | `1` | No beam search |
| `temperature` | `1.0` | Inert under greedy |
| `top_k` | `1` | Inert under greedy |
| `top_p` | `1.0` | Inert under greedy |
| `repetition_penalty` | `1.0` | No penalty |
| `no_repeat_ngram_size` | `0` | Disabled |
| `max_new_tokens` | `64` | Max generated tokens per segment (the only length control we set) |

In [ ]:
from pathlib import Path

from chirpe.data.summarizer import Phi3OnnxSummarizer

# The notebook may run from notebooks/ or the repo root, so try both prefixes.
PHI3_MODEL_DIR_CANDIDATES = [
    Path("outputs/local_onnx_llm/phi3-mini-4k-instruct-onnx/cpu_and_mobile/cpu-int4-rtn-block-32-acc-level-4"),
    Path("../outputs/local_onnx_llm/phi3-mini-4k-instruct-onnx/cpu_and_mobile/cpu-int4-rtn-block-32-acc-level-4"),
]
PHI3_MODEL_DIR = next((p for p in PHI3_MODEL_DIR_CANDIDATES if p.exists()), None)

# Tokenizer backend: "og" (onnxruntime-genai, default) or "hf" (transformers
# AutoTokenizer). og.Generator runs the model in both cases; only the tokenizer
# changes. They produce identical token IDs on clinical text, so the choice does
# not change outputs (see scripts/experiments/tokenizer_parity_check.py). "hf" is
# the deployable option, since onnxruntime-genai has no Triton support.
TOKENIZER_BACKEND = "og"

summarizer = Phi3OnnxSummarizer(
    model_dir=str(PHI3_MODEL_DIR) if PHI3_MODEL_DIR else None,
    max_new_tokens=64,
    download=PHI3_MODEL_DIR is None,
    tokenizer_backend=TOKENIZER_BACKEND,
)

print(f"Loaded Phi-3 ONNX summarizer (tokenizer_backend={TOKENIZER_BACKEND})")

In [ ]:
# Generation settings used for every segment, pinned explicitly in
# Phi3OnnxSummarizer.summarize_segment (set via og GeneratorParams.set_search_options).
# Decoding is greedy and deterministic; the sampling knobs are inert under greedy
# and the repetition controls are disabled. max_new_tokens is the only length control.
generation_settings = {
    "do_sample": False,            # greedy decoding (deterministic)
    "num_beams": 1,                # no beam search
    "temperature": 1.0,            # inert under greedy
    "top_k": 1,                    # inert under greedy
    "top_p": 1.0,                  # inert under greedy
    "repetition_penalty": 1.0,     # no penalty
    "no_repeat_ngram_size": 0,     # disabled
    "max_new_tokens": summarizer.max_new_tokens,
}
generation_settings

## Inside ReadyAI: Load CHiRPE Model

ReadyAI loads the CHiRPE classifier. Change `MODEL_PATH` to test a different
checkpoint.

In [6]:
from chirpe.models.classifier import CHRClassifier

MODEL_PATH_CANDIDATES = [
    Path("outputs/string_onnx_checkpoints/bert/best_model"),
    Path("../outputs/string_onnx_checkpoints/bert/best_model"),
]
MODEL_PATH = next((path for path in MODEL_PATH_CANDIDATES if path.exists()), MODEL_PATH_CANDIDATES[0])

if not MODEL_PATH.exists():
    raise FileNotFoundError(
        f"Model checkpoint not found: {MODEL_PATH}. Update MODEL_PATH to a trained CHiRPE best_model directory."
    )

classifier = CHRClassifier(model_name=str(MODEL_PATH))
classifier.load(MODEL_PATH)

print(f"Loaded CHiRPE model from: {MODEL_PATH}")

Loaded CHiRPE model from: ../outputs/string_onnx_checkpoints/bert/best_model


## Inside ReadyAI: Summarize (ONNX) then Predict per Segment

ReadyAI summarizes each segment's raw `text` with the Phi-3 ONNX model, then runs
`classifier.predict()` on the generated summaries to produce per-segment results.
Transcript-level aggregation is applied in a later step (also inside ReadyAI), so
this function keeps the segment-level outputs separate rather than calling
`predict_with_segments()`.

In [7]:
LABELS = {0: "Healthy", 1: "CHR-P"}


def summarize_and_predict_payload(summarizer, classifier, payload, text_field="text"):
    """Run ONNX summarization + CHiRPE on pre-segmented raw text, per segment only."""
    segments = payload.get("segments", [])
    if not segments:
        raise ValueError("Payload must contain a non-empty 'segments' list.")

    summaries = []
    for index, segment in enumerate(segments):
        text = segment.get(text_field)
        if not text:
            raise ValueError(f"Segment {index} is missing required text field: {text_field!r}")
        summaries.append(summarizer.summarize_segment(text))

    predictions, probabilities = classifier.predict(summaries, return_probs=True)

    segment_outputs = []
    for index, (segment, summary, prediction, probability) in enumerate(
        zip(segments, summaries, predictions, probabilities)
    ):
        prediction = int(prediction)
        healthy_prob = float(probability[0])
        chrp_prob = float(probability[1])

        segment_outputs.append(
            {
                "segment_id": segment.get("segment_id", f"segment_{index + 1}"),
                "domain": segment.get("domain", "unknown"),
                "summary": summary,
                "prediction": prediction,
                "label": LABELS[prediction],
                "confidence": float(max(healthy_prob, chrp_prob)),
                "probabilities": {
                    "Healthy": healthy_prob,
                    "CHR-P": chrp_prob,
                },
            }
        )

    return {
        "participant_id": payload.get("participant_id", "unknown"),
        "segments": segment_outputs,
    }

In [8]:
readyai_output = summarize_and_predict_payload(summarizer, classifier, segmented_payload)
readyai_output

{'participant_id': 'demo_001',
 'segments': [{'segment_id': 'demo_001_seg_001',
   'domain': 'P1_Unusual_Thoughts',
   'summary': 'The patient reports experiencing significant personal resonance with media content, suggesting possible delusional thinking. This symptom may impact daily functioning and warrants further assessment for potential psychotic disorders.',
   'prediction': 1,
   'label': 'CHR-P',
   'confidence': 0.6322894096374512,
   'probabilities': {'Healthy': 0.36771050095558167,
    'CHR-P': 0.6322894096374512}},
  {'segment_id': 'demo_001_seg_002',
   'domain': 'P2_Suspiciousness',
   'summary': 'The interviewee expresses concern about potential surveillance, with intermittent episodes of worry about being watched by strangers. This anxiety may impact their daily functioning and warrants further assessment for risk-related behaviors.',
   'prediction': 1,
   'label': 'CHR-P',
   'confidence': 0.5646314024925232,
   'probabilities': {'Healthy': 0.4353685975074768,
    'CH

## Per-Segment Results (Intermediate, Inside ReadyAI)

These are ReadyAI's per-segment results: one model result per input segment,
each including the ONNX-generated `summary`. This is an intermediate stage — the
transcript-level prediction is produced by the aggregation step that follows,
which is also inside ReadyAI.

In [9]:
print(json.dumps(readyai_output, indent=2))

{
  "participant_id": "demo_001",
  "segments": [
    {
      "segment_id": "demo_001_seg_001",
      "domain": "P1_Unusual_Thoughts",
      "summary": "The patient reports experiencing significant personal resonance with media content, suggesting possible delusional thinking. This symptom may impact daily functioning and warrants further assessment for potential psychotic disorders.",
      "prediction": 1,
      "label": "CHR-P",
      "confidence": 0.6322894096374512,
      "probabilities": {
        "Healthy": 0.36771050095558167,
        "CHR-P": 0.6322894096374512
      }
    },
    {
      "segment_id": "demo_001_seg_002",
      "domain": "P2_Suspiciousness",
      "summary": "The interviewee expresses concern about potential surveillance, with intermittent episodes of worry about being watched by strangers. This anxiety may impact their daily functioning and warrants further assessment for risk-related behaviors.",
      "prediction": 1,
      "label": "CHR-P",
      "confidenc

## Inside ReadyAI: Aggregate Segment Outputs

This cell performs transcript-level aggregation **inside ReadyAI**. It consumes
the per-segment results and produces the participant-level CHR-P decision that
ReadyAI returns.

In [ ]:
def readyai_majority_vote(segment_level_output):
    """Transcript-level majority vote, applied inside ReadyAI."""
    votes = [segment["prediction"] for segment in segment_level_output["segments"]]
    chrp_votes = sum(vote == 1 for vote in votes)
    healthy_votes = sum(vote == 0 for vote in votes)
    final_prediction = 1 if chrp_votes > healthy_votes else 0
    return {
        "participant_id": segment_level_output["participant_id"],
        "prediction": final_prediction,
        "label": LABELS[final_prediction],
        "chrp_votes": chrp_votes,
        "healthy_votes": healthy_votes,
    }

aggregated_output = readyai_majority_vote(readyai_output)
aggregated_output

## Summary

The whole pipeline runs inside ReadyAI:

- Ingest the raw transcript JSON.
- Segment the transcript into PSYCHS-domain segments.
- Summarize each segment with the Phi-3 ONNX model.
- Run CHiRPE on each segment summary.
- Aggregate the per-segment results into a transcript-level CHR-P decision.

ReadyAI takes the raw transcript JSON as input and returns the aggregated
participant-level result.